In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

CATALOG = "gwd_de"
source_table = f"{CATALOG}.bronze.customers"
target_table = f"{CATALOG}.silver.customers"

In [0]:
df_bronze = spark.table(source_table)

df_silver = df_bronze.select(
    F.col("customer_id"),
    F.col("customer_name"),
    F.initcap(F.col("customer_type")).alias("customer_type"),
    F.when(F.col("country").isNull(), "Unknown")
     .when(F.col("country") == "U.S.A", "USA")
     .otherwise(F.initcap(F.col("country"))).alias("country"),
    F.col("signup_date")
)

df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)